# Growing Neural Cellular Automata with Reinforcement Learning [![Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxencefaldor/cax/blob/main/examples/63_growing_nca_rl.ipynb)

Growing NCA is usually trained by backpropagating through the whole developmental process,
which costs time and memory in proportion to its length.

Here the system is treated as a policy $\pi_\theta$ mapping a state to the next one, and every
state is scored by its negative distance to the target, $r(s) = -\lVert s_\text{RGBA} - y \rVert^2$.
The objective is the discounted return of a rollout of length $H$, with everything beyond the
horizon summarized by a learned value function $V$:

$$J(\theta) = \mathbb{E}\Big[\sum_{t=0}^{H-1} \gamma^{t} \, r(s_t) \; + \; \gamma^{H} V(s_H)\Big]$$

Taking $H$ well below the full number of developmental steps is what makes this cheap, since
cost scales with $H$ rather than with the length of the process. The cellular automaton is
differentiable, so $\nabla_\theta J$ flows analytically through the rollout and through
$V(s_H)$, instead of being estimated from samples as a policy gradient would be.

## Installation

You will need Python 3.12 or later, and a working JAX installation. For example, you can install JAX with:

In [ ]:
%pip install -U "jax[cuda]"

Then, install CAX from PyPi:

In [ ]:
%pip install -U "cax[examples]"

## Import

In [ ]:
import time

import jax
import jax.numpy as jnp
import mediapy
import optax
from flax import nnx
from jax import Array

from cax.core import ComplexSystem
from cax.core.perceive import ConvPerceive, grad_kernel, identity_kernel
from cax.core.update import NCAUpdate
from cax.nn.pool import Pool
from cax.utils import clip_and_uint8, get_emoji_array, rgba_to_rgb

## Configuration

In [ ]:
seed = 0

channel_size = 16
num_kernels = 3
hidden_size = 128
cell_dropout_rate = 0.5

num_steps = 128
horizon = 64
discount = 0.999
lambda_ = 0.95

critic_features = (32, 64, 128)
critic_hidden_size = 128
critic_learning_rate = 3e-4
critic_num_updates = 4
target_step_size = 0.05
bootstrap_warmup = 200

pool_size = 1_024
batch_size = 32
num_resets = 16
num_train_steps = 16_384
learning_rate = 3e-3

emoji = "🦎"
size = 40
pad_width = 16

key = jax.random.key(seed)
rngs = nnx.Rngs(seed)

## Dataset

In [ ]:
y = get_emoji_array(emoji, size, pad_width)

mediapy.show_image(y)

## Instantiate system

In [ ]:
class GrowingNCA(ComplexSystem):
    """Growing Neural Cellular Automata class."""

    def __init__(self, *, rngs: nnx.Rngs):
        """Initialize Growing NCA.

        Args:
            rngs: rng key.

        """
        self.perceive = ConvPerceive(
            channel_size=channel_size,
            perception_size=num_kernels * channel_size,
            feature_group_count=channel_size,
            rngs=rngs,
        )
        self.update = NCAUpdate(
            channel_size=channel_size,
            perception_size=num_kernels * channel_size,
            hidden_layer_sizes=(hidden_size,),
            cell_dropout_rate=cell_dropout_rate,
            zeros_init=True,
            rngs=rngs,
        )

        # Initialize kernel with sobel filters
        kernel = jnp.concatenate(
            [identity_kernel(num_dims=2), grad_kernel(num_dims=2)], axis=-1
        )
        kernel = jnp.expand_dims(
            jnp.concatenate([kernel] * channel_size, axis=-1), axis=-2
        )
        self.perceive.conv.kernel[...] = kernel

    def _step(self, state: Array, input: Array | None = None) -> Array:
        perception = self.perceive(state)
        next_state = self.update(state, perception, input)

        return next_state

    @nnx.jit
    def render(self, state):
        """Render state to RGB."""
        rgba = state[..., -4:]
        rgb = rgba_to_rgb(rgba)

        # Clip values to valid range and convert to uint8
        return clip_and_uint8(rgb)

    @nnx.jit
    def render_rgba(self, state):
        """Render state to RGBA."""
        rgba = state[..., -4:]

        # Clip values to valid range and convert to uint8
        return clip_and_uint8(rgba)

In [ ]:
cs = GrowingNCA(rngs=rngs)

In [ ]:
params = nnx.state(cs, nnx.Param)
print("Number of params:", sum(x.size for x in jax.tree.leaves(params)))

## Instantiate critic

The critic estimates the return of a state, which is what lets the policy be trained without
unrolling the whole developmental process.

Every reward is a negative distance, so no state can have positive value. Writing
$V_\phi = -\operatorname{softplus}(\cdot)$ builds that into the critic and keeps the policy from
chasing states where an unconstrained critic would extrapolate upwards. Bootstrap values come
from a slowly moving copy $V_{\phi^-}$, updated as $\phi^- \leftarrow (1 - \tau)\,\phi^- + \tau\,\phi$,
so the target the critic regresses toward does not move with every gradient step.

In [ ]:
class Critic(nnx.Module):
    """Critic estimating the return of a state."""

    def __init__(self, *, features: tuple[int, ...], hidden_size: int, rngs: nnx.Rngs):
        """Initialize the critic.

        Args:
            features: Number of features of each strided convolution.
            hidden_size: Size of the hidden layer.
            rngs: rng key.

        """
        self.convs = nnx.List(
            [
                nnx.Conv(
                    in_features=in_features,
                    out_features=out_features,
                    kernel_size=(3, 3),
                    strides=(2, 2),
                    padding="SAME",
                    rngs=rngs,
                )
                for in_features, out_features in zip(
                    (channel_size, *features[:-1]), features, strict=True
                )
            ]
        )
        self.linear_1 = nnx.Linear(features[-1], hidden_size, rngs=rngs)
        self.linear_2 = nnx.Linear(hidden_size, 1, rngs=rngs)

    def __call__(self, state: Array) -> Array:
        """Estimate the value of a state."""
        x = state
        for conv in self.convs:
            x = jax.nn.relu(conv(x))

        # Pool over space so the head does not depend on the size of the grid
        x = jnp.mean(x, axis=(-3, -2))
        x = jax.nn.relu(self.linear_1(x))
        value = jnp.squeeze(self.linear_2(x), axis=-1)

        # Rewards are negative distances, so values are negative too
        return -jax.nn.softplus(value)

In [ ]:
critic = Critic(features=critic_features, hidden_size=critic_hidden_size, rngs=rngs)

# A slow-moving copy of the critic provides every bootstrap value, so that the
# critic never regresses toward its own moving predictions.
critic_target = nnx.clone(critic)

## Sample initial state

In [ ]:
def sample_state():
    """Sample a state with a single alive cell."""
    spatial_dims = y.shape[:2]

    # Init state
    state = jnp.zeros((*spatial_dims, channel_size))

    # Set the center cell to alive, with hidden channels at one
    mid = tuple(size // 2 for size in spatial_dims)
    state = state.at[mid[0], mid[1], -1].set(1.0)
    return state.at[mid[0], mid[1], :-4].set(1.0)

## Train

### Pool

In [ ]:
state = jax.vmap(lambda _: sample_state())(jnp.zeros(pool_size))

pool = Pool.create({"state": state})

### Optimizer

In [ ]:
lr_sched = optax.warmup_cosine_decay_schedule(
    init_value=0.01 * learning_rate,
    peak_value=learning_rate,
    warmup_steps=200,
    decay_steps=num_train_steps,
    end_value=0.02 * learning_rate,
)


def normalize_by_norm(eps: float = 1e-8) -> optax.GradientTransformation:
    """Normalize each gradient tensor by its own norm."""

    def init_fn(params):
        del params
        return optax.EmptyState()

    def update_fn(updates, state, params=None):
        del params
        return jax.tree.map(lambda g: g / (jnp.linalg.norm(g) + eps), updates), state

    return optax.GradientTransformation(init_fn, update_fn)


optimizer = optax.chain(
    optax.zero_nans(),
    normalize_by_norm(),
    optax.clip_by_global_norm(0.5),
    optax.adam(learning_rate=lr_sched),
)

update_params = nnx.All(nnx.Param, nnx.PathContains("update"))
optimizer = nnx.Optimizer(cs, optimizer, wrt=update_params)

critic_optimizer = nnx.Optimizer(
    critic,
    optax.chain(
        optax.clip_by_global_norm(1.0),
        optax.adam(learning_rate=critic_learning_rate),
    ),
    wrt=nnx.Param,
)

### Reward

In [ ]:
def mse(state):
    """Mean Squared Error."""
    return jnp.mean(jnp.square(state[..., -4:] - y))


def reward_fn(state):
    """Reward a state for being close to the target."""
    return -mse(state)

### Return

The return of a rollout mixes the rewards it collected with the value of the state it ended in.
The $\lambda$-return does this recursively, backwards from $G_H = V_{\phi^-}(s_H)$:

$$G_t = r_t + \gamma \big[(1 - \lambda) \, V_{\phi^-}(s_{t+1}) + \lambda \, G_{t+1}\big]$$

so $\lambda = 0$ trusts a single step and its bootstrap, $\lambda = 1$ trusts the whole rollout,
and values in between interpolate.

In [ ]:
def lambda_return(rewards, values):
    """Compute the lambda-return of a trajectory.

    Args:
        rewards: Rewards of shape `(horizon,)`, collected on entering each state.
        values: Estimated values of shape `(horizon + 1,)`, of the initial state
            through the final state of the rollout.

    Returns:
        The lambda-returns of shape `(horizon,)`, one for each state but the last.

    """

    def scan_fn(carry, x):
        reward, value = x
        carry = reward + discount * ((1.0 - lambda_) * value + lambda_ * carry)
        return carry, carry

    _, returns = jax.lax.scan(scan_fn, values[-1], (rewards, values[1:]), reverse=True)
    return returns

### Loss

In [ ]:
def rollout(cs, state):
    """Roll the system out for `horizon` steps and reward every state it visits."""
    state_axes = nnx.StateAxes({nnx.RngState: 0, ...: None})
    _, states = nnx.split_rngs(splits=batch_size)(
        nnx.vmap(
            lambda cs, state: cs(state, num_steps=horizon, return_states=True),
            in_axes=(state_axes, 0),
        )
    )(cs, state)

    rewards = jax.vmap(jax.vmap(reward_fn))(states)
    return states, rewards


def actor_loss_fn(cs, critic_target, state, bootstrap_scale):
    """Compute the negative bootstrapped return of a rollout.

    The gradient flows analytically through the rollout, and through the value of
    the state it ends in, which is what carries credit past the horizon.
    `bootstrap_scale` ramps up early in training so that the policy is not steered
    by the critic before the critic is accurate.
    """
    states, rewards = rollout(cs, state)

    returns = jnp.sum(rewards * discount ** jnp.arange(horizon), axis=-1)
    returns = returns + bootstrap_scale * discount**horizon * critic_target(
        states[:, -1]
    )

    return -jnp.mean(returns), (states, rewards)


def critic_loss_fn(critic, critic_target, states, rewards):
    """Regress the critic toward the lambda-returns of a rollout."""
    targets = jax.vmap(lambda_return)(rewards, critic_target(states))

    return jnp.mean(jnp.square(jax.lax.stop_gradient(targets) - critic(states[:, :-1])))

### Train step

In [ ]:
@nnx.jit
def train_step(
    cs, critic, critic_target, optimizer, critic_optimizer, pool, key, scale
):
    """Train step."""
    # Sample from pool
    pool_idx, batch = pool.sample(key, batch_size=batch_size, replace=False)
    current_state = batch["state"]

    # Sort by descending loss
    sort_idx = jnp.argsort(jax.vmap(mse)(current_state), descending=True)
    pool_idx = pool_idx[sort_idx]
    current_state = current_state[sort_idx]

    # Sample new states to replace the worst
    new_state = sample_state()
    current_state = current_state.at[:num_resets].set(new_state)

    # Update the policy through the rollout
    (loss, (states, rewards)), grad = nnx.value_and_grad(
        actor_loss_fn, has_aux=True, argnums=nnx.DiffState(0, update_params)
    )(cs, critic_target, current_state, scale)
    optimizer.update(cs, grad)

    # Fit the critic on the states the rollout just visited
    states = jnp.concatenate([current_state[:, None], states], axis=1)
    for _ in range(critic_num_updates):
        critic_loss, critic_grad = nnx.value_and_grad(critic_loss_fn)(
            critic, critic_target, states, rewards
        )
        critic_optimizer.update(critic, critic_grad)

    # Let the target critic drift toward the critic
    nnx.update(
        critic_target,
        jax.tree.map(
            lambda t, p: (1.0 - target_step_size) * t + target_step_size * p,
            nnx.state(critic_target, nnx.Param),
            nnx.state(critic, nnx.Param),
        ),
    )

    pool = pool.update(pool_idx, {"state": states[:, -1]})
    return loss, critic_loss, pool

### Main loop

In [ ]:
print_interval = 128

losses = []
start = time.perf_counter()
for i in range(num_train_steps):
    key, subkey = jax.random.split(key)
    scale = jnp.minimum(i / bootstrap_warmup, 1.0)
    loss, critic_loss, pool = train_step(
        cs, critic, critic_target, optimizer, critic_optimizer, pool, subkey, scale
    )

    losses.append(loss)
    if i % print_interval == 0 or i == num_train_steps - 1:
        avg_loss = sum(losses[-print_interval:]) / len(losses[-print_interval:])
        elapsed = time.perf_counter() - start
        pool_mse = jnp.mean(jax.vmap(mse)(pool.data["state"]))
        print(
            f"Step {i:>4}/{num_train_steps} | {elapsed:6.1f}s "
            f"| Loss {avg_loss:.3e} | Critic {critic_loss:.2e} "
            f"| Pool MSE {pool_mse:.3e}"
        )

print(f"✨ Trained for {num_train_steps} steps in {time.perf_counter() - start:.0f}s")

## Run

In [ ]:
num_examples = 8

state_init = jax.vmap(lambda _: sample_state())(jnp.zeros(num_examples))

state_axes = nnx.StateAxes({nnx.RngState: 0, ...: None})
state_final, states = nnx.split_rngs(splits=num_examples)(
    nnx.vmap(
        lambda cs, state_init: cs(state_init, num_steps=num_steps, return_states=True),
        in_axes=(state_axes, 0),
    )
)(cs, state_init)

## Visualize

In [ ]:
frames_final = nnx.vmap(
    lambda cs, state: cs.render(state),
    in_axes=(None, 0),
)(cs, state_final)
frames_final_rgba = nnx.vmap(
    lambda cs, state: cs.render_rgba(state),
    in_axes=(None, 0),
)(cs, state_final)

mediapy.show_images(frames_final.repeat(2, axis=-3).repeat(2, axis=-2))
mediapy.show_images(frames_final_rgba.repeat(2, axis=-3).repeat(2, axis=-2))

In [ ]:
states = jnp.concatenate([state_init[:, None], states], axis=1)
frames = nnx.vmap(
    lambda cs, states: cs.render(states),
    in_axes=(None, 0),
)(cs, states)

mediapy.show_videos(frames.repeat(2, axis=-3).repeat(2, axis=-2))